In [ ]:
import os

from trainer import Trainer, TrainerArgs

from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits
from TTS.utils.manage import ModelManager

# -----------------------
# PATHS
# -----------------------
BASE_DIR = os.getcwd()
DATASET_PATH = os.path.join(BASE_DIR, "dataset")
OUTPUT_PATH = os.path.join(BASE_DIR, "bn_vits_output")

# -----------------------
# LOAD PRETRAINED MODEL
# -----------------------
manager = ModelManager()

# This downloads the model and provides paths to the weights and config
model_path, config_path, _ = manager.download_model(
    "tts_models/bn/custom/vits-male"
)

print("Pretrained model:", model_path)
print("Config:", config_path)

# -----------------------
# DATASET CONFIG
# -----------------------
dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    meta_file_train="metadata.csv",
    path=DATASET_PATH,
    language="bn"
)

# -----------------------
# VITS CONFIG (FINE-TUNE MODE)
# -----------------------
# FIX 1: Load the original config to ensure audio/model architecture matches perfectly
config = VitsConfig()
config.load_json(config_path)

# Overwrite strictly the training-specific parameters
config.run_name = "bangla_vits_finetune"
config.batch_size = 32
config.eval_batch_size = 16
config.num_loader_workers = 4
config.num_eval_loader_workers = 2
config.run_eval = True
config.test_delay_epochs = 5
config.epochs = 20
config.save_step = 500
config.print_step = 25
config.print_eval = True
config.mixed_precision = True
config.cudnn_benchmark = False

# Attach the dataset and define the output folder
config.output_path = OUTPUT_PATH
config.datasets = [dataset_config]

# -----------------------
# LOAD DATA
# -----------------------
# FIX: Pass the entire 'config' object, NOT 'config.datasets'
train_samples, eval_samples = load_tts_samples(
    config,
    eval_split=True,
    eval_split_size=0.05,
)

# -----------------------
# INIT MODEL
# -----------------------
model = Vits(config)

print(config.to_json())

# -----------------------
# TRAINER
# -----------------------
# FIX 4: Pass the model_path directly to TrainerArgs via `restore_path`
trainer = Trainer(
    TrainerArgs(restore_path=str(model_path)),  # <--- WRAP IN str()
    config,
    OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

Pretrained model: /home/khansun/.local/share/tts/tts_models--bn--custom--vits-male/model_file.pth
Config: /home/khansun/.local/share/tts/tts_models--bn--custom--vits-male/config.json


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 24
 | > Num. of Torch Threads: 12
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/home/khansun/repos/coqui-ai-TTS/bn_vits_output/bangla_vits_finetune-June-06-2026_01+05PM-474faf46


{
    "output_path": "/home/khansun/repos/coqui-ai-TTS/bn_vits_output",
    "logger_uri": null,
    "run_name": "bangla_vits_finetune",
    "project_name": null,
    "run_description": "\ud83d\udc38Coqui trainer run.",
    "print_step": 25,
    "plot_step": 100,
    "model_param_stats": false,
    "wandb_entity": null,
    "dashboard_logger": "tensorboard",
    "save_on_interrupt": true,
    "log_model_step": null,
    "save_step": 500,
    "save_n_checkpoints": 5,
    "save_checkpoints": true,
    "save_all_best": true,
    "save_best_after": 500,
    "target_loss": null,
    "print_eval": true,
    "test_delay_epochs": 5,
    "run_eval": true,
    "run_eval_steps": null,
    "distributed_backend": "nccl",
    "distributed_url": "tcp://localhost:54321",
    "mixed_precision": true,
    "precision": "fp16",
    "epochs": 20,
    "batch_size": 32,
    "eval_batch_size": 16,
    "grad_clip": [
        1000.0,
        1000.0
    ],
    "scheduler_after_epoch": false,
    "lr": 0.001,
    

 > Restoring from model_file.pth ...
 > Restoring Model...
 > Model restored from step 910000

 > Model has 83050732 parameters

 > EPOCH: 0/19
 --> /home/khansun/repos/coqui-ai-TTS/bn_vits_output/bangla_vits_finetune-June-06-2026_01+05PM-474faf46

 > TRAINING (2026-06-06 13:06:23) 
শ্বশুড়বাড়িতে এলাম।

বল কী?

তাই বিল আমরাই দেব।

জ্বি।

Character '\n' not found in the vocabulary. Discarding it.
Character '\n' not found in the vocabulary. Discarding it.
Character '\n' not found in the vocabulary. Discarding it.
Character '\n' not found in the vocabulary. Discarding it.
কী দরকার–।

Character '–' not found in the vocabulary. Discarding it.
–কাঁদে কে চাচি?

Character '–' not found in the vocabulary. Discarding it.
আমাকে বললেন…  ।

Character '…' not found in the vocabulary. Discarding it.
‘কী করেছিলে তুমি?”

Character '”' not found in the vocabulary. Discarding it.
তাই আমি এলে–।

Character '–' not found in the vocabulary. Discarding it.
/home/khansun/miniconda3/envs/tts/lib/python3.14/sit

In [ ]:
! tts --text "সোর্স অব ফান্ড হিসেবে বাসা ভাড়া, দোকান ভাড়া বা ফিশারি লিজ দেখালে কী কী ডকুমেন্ট জমা দিতে হবে?" \
    --model_path "bn_vits_output/bangla_vits_finetune-June-06-2026_01+05PM-474faf46/best_model.pth" \
    --config_path "bn_vits_output/bangla_vits_finetune-June-06-2026_05+16AM-474faf46/config.json" \
    --out_path "out.wav"

Using model: vits
Setting up Audio Processor...
Language `bn` not supported by pySBD, using English for sentence splitting.
Segmenters initialized for: dict_keys(['bn'])
Text: সোর্স অব ফান্ড হিসেবে বাসা ভাড়া, দোকান ভাড়া বা ফিশারি লিজ দেখালে কী কী ডকুমেন্ট জমা দিতে হবে?
Splitting into sentences (language: bn).
Input: ['সোর্স অব ফান্ড হিসেবে বাসা ভাড়া, দোকান ভাড়া বা ফিশারি লিজ দেখালে কী কী ডকুমেন্ট জমা দিতে হবে?']
Processing time: 1.159
Real-time factor: 0.169
Saved TTS output to out.wav


In [26]:
from IPython.display import Audio

# Load and display the audio player widget
Audio(filename="out.wav")